In [1]:
import numpy as np

def deletion_suffix_channel(x):
    n = len(x)
    L = np.random.randint(0, n + 1)  # suffix length
    prefix = x[:n - L]
    noise = np.random.randint(0, 2, size=L)
    return np.concatenate([prefix, noise])

In [2]:
def generate_traces_max(x, N_max):
    return [deletion_suffix_channel(x) for _ in range(N_max)]

In [3]:
def estimate_x_fast(traces):
    max_len = max(len(t) for t in traces)
    
    mat = np.zeros((len(traces), max_len), dtype=np.int8)
    mask = np.zeros_like(mat)

    for i, t in enumerate(traces):
        L = len(t)
        mat[i, :L] = t
        mask[i, :L] = 1

    ones = mat.sum(axis=0)
    counts = mask.sum(axis=0)

    # avoid division, just compare
    return (ones > counts // 2).astype(np.int8)

In [4]:
def estimate_error_adaptive(n, N, delta=0.01, max_trials=1000):
    errors = 0

    for t in range(1, max_trials + 1):
        X = np.random.randint(0, 2, size=n)
        traces = generate_traces_max(X, N)  # reuse N only here
        
        if not np.array_equal(X, estimate_x_fast(traces)):
            errors += 1

        err = errors / t

        if t > 0.4 * max_trials and err < delta / 2:
            return err

    return errors / max_trials

In [5]:
def find_min_N(n, delta=0.01, N_max=5000):
    low, high = 1, N_max

    while low < high:
        mid = (low + high) // 2
        
        err = estimate_error_adaptive(n, mid, delta)

        if err < delta:
            high = mid
        else:
            low = mid + 1

    return low

In [6]:
def initial_guess(n, delta):
    return int(2 * np.log(n / delta))

In [7]:
def find_min_N_fast(n, delta=0.01):
    # N0 = initial_guess(n, delta)
    low =  256
    err=1
    high = 256
    
    while err > delta:
        low=high
        high*=2
        err=estimate_error_adaptive(n, high, delta)
        print("high=",high,"err=",err)
        
    while low < high:
        mid = (low + high) // 2
        err = estimate_error_adaptive(n, mid, delta)
        print(f"Testing: n={n}, N={mid}, error={err:.4f}")
        if err < delta:
            high = mid
        else:
            low = mid + 1

    return low

In [ ]:
n_val=range(50,1000,50)
# n_val=[1000,1050]

for n in n_val:
    N_estimated = find_min_N_fast(n, delta=0.01)
    print(f"Estimated minimum N for n={n}: {N_estimated}")

high= 512 err= 0.526
high= 1024 err= 0.366
high= 2048 err= 0.226
high= 4096 err= 0.094
high= 8192 err= 0.039
high= 16384 err= 0.0
Testing: n=50, N=12288, error=0.0130
Testing: n=50, N=14336, error=0.0025
Testing: n=50, N=13312, error=0.0100
Testing: n=50, N=13824, error=0.0150
Testing: n=50, N=14080, error=0.0070
Testing: n=50, N=13952, error=0.0130
Testing: n=50, N=14016, error=0.0120
Testing: n=50, N=14048, error=0.0070
Testing: n=50, N=14032, error=0.0090
Testing: n=50, N=14024, error=0.0050
Testing: n=50, N=14020, error=0.0080
Testing: n=50, N=14018, error=0.0050
Testing: n=50, N=14017, error=0.0080
Estimated minimum N for n=50: 14017
high= 512 err= 0.812
high= 1024 err= 0.701
high= 2048 err= 0.549
high= 4096 err= 0.341
high= 8192 err= 0.188
high= 16384 err= 0.109
high= 32768 err= 0.03
high= 65536 err= 0.007
Testing: n=100, N=49152, error=0.0190
Testing: n=100, N=57344, error=0.0100
Testing: n=100, N=61440, error=0.0050
Testing: n=100, N=59392, error=0.0050
Testing: n=100, N=58368,

In [ ]:
delta_val=range(0.01,0.06,0.01)
# n_val=[1000,1050]
n=1000
for delta in delta_val:
    N_estimated = find_min_N_fast(n, delta)
    print(f"Estimated minimum N for n={n}: {N_estimated}")